# In-Class Challenge Assignment: Experimenting with the Perceptron
# Day 19 Extension
# CMSE 202

## Now that you have a working Perceptron Classifier... let's experiment with it a bit!

When building and testing your Perceptron Classifier you used a simplified version of the iris dataset that has been reduced to just two features and two class labels. 

### Will your Perceptron classifier work on a more complex dataset?

Another widely used dataset for experimenting with binary classification is the [sonar dataset](https://archive.ics.uci.edu/ml/datasets/Connectionist+Bench+(Sonar,+Mines+vs.+Rocks)).

A version of this dataset can be found here:

`https://raw.githubusercontent.com/msu-cmse-courses/cmse202-supplemental-data/main/data/sonar_pca10_attr.csv`

Make sure you take a moment to read the [UC Irvine Machine Learning Repository page](https://archive.ics.uci.edu/ml/datasets/Connectionist+Bench+(Sonar,+Mines+vs.+Rocks)) to understand exactly what is in this dataset, but essentially is a collection of sonar measurements of rocks and "mines" (metal cynlinders). We are using a version of the dataset that has been reduced to 10 principal components (instead of the original 60 attributes) to make it easier to work with. The class labels are still the same as in the original dataset, and are located in the last column of the dataset.



---
### Testing your new tool and exploring others

In class today, try to accomplish the following:

1. Load up the sonar dataset and change the class labels so that they can be used with the Perceptron classifier.

2. Use the Perceptron classifier you built from scratch to see how well you can do at distinguishing rocks from mines. You may need to make some modifications to your code if you didn't build it to be flexible enough to accept an arbitary number of data deatures. Experiment with the learning rate and number of iterations to see how high of an accuracy you can get with your classifier.

3. If you get your Perceptron classifier working, can you figure out how to use the Perceptron Classifier that is available in [scikit-learn](https://scikit-learn.org/stable/index.html)? You may need to do a bit of Google searching and exploration of the documentation to figure this out. How well does the scikit-learn version do compared to the one you built?

<!--
4. If you're feeling really ambitious, can you build a Perceptron classifier with [Tensorflow](https://www.tensorflow.org/)? Remember, the Perceptron is basically just a single-neuron single-layer neural network. This requires installation of Tensorflow and relevant APIs.
-->

4. The logistic regression model (Day 15) is also a multi-variable classifier. Use it on the same dataset. Compare the results of your perceptron classifier against that obtained from and discuss your observations.

---
&#9989; **Do This**: Load the sonar data.  Make a variable with the features and a variable with the labels.  

In [47]:
import pandas as pd
import numpy as np

data = pd.read_csv("sonar_pca10_attr.csv")

X = data.iloc[:, :-1].values

labels = data.iloc[:, -1].astype(str).str.strip()

y = np.where(labels == "Mine", 1, -1)


Copy your percentron class to the cell below. **Note** sklearn has a **Perceptron** function. We should avoid using the same function name of your perceptron class. 

**Note:** You may need to modify your code to take the new label definitions into account.

In [71]:

class MyPerceptron:

    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.lr = learning_rate
        self.n_iters = n_iters

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            for i in range(n_samples):

                linear_output = np.dot(X[i], self.weights) + self.bias
                y_pred = 1 if linear_output >= 0 else -1

                if y[i] * y_pred <= 0:
                    self.weights = self.weights + self.lr * y[i] * X[i]
                    self.bias = self.bias + self.lr * y[i]

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return np.where(linear_output >= 0, 1, -1)

Train your percentron class with sonar data. What's the accuracy?

In [73]:
# put your code here
model = MyPerceptron(learning_rate=0.001, n_iters=500)

model.fit(X, y)

predictions = model.predict(X)

accuracy = np.mean(predictions == y)

print("Accuracy:", accuracy)

Accuracy: 0.7548076923076923


---
&#9989; **Do This**: Use the **Perceptron** function from sklearn library to classify the same dataset in the cell below. Compared to your perceptron classifier, how is the performance of the perceptron in the sklearn library?

In [75]:
# put your code here
from sklearn.linear_model import Perceptron
import numpy as np

sk_model = Perceptron(max_iter=1000, eta0=0.01, tol=1e-3, random_state=0)
sk_model.fit(X, y)

sk_predictions = sk_model.predict(X)
sk_accuracy = np.mean(sk_predictions == y)

print("Sklearn Perceptron Accuracy:", sk_accuracy)

Sklearn Perceptron Accuracy: 0.7259615384615384


---
&#9989; **Do This**: Use **logistic regression model** from statsmodel library to classify the same dataset in the cell below. 

* Reminder that we will need add a constant to the input features, which is equivalent to the bias weight.
* The Logit function requires the labels to be different values than the current `[-1, 1]` values (**WHY??**- Discuss with your group). You'll need to replace the `-1` label to make it work. 
* Let's set the test set size to 15% in the train-test split, and fit the model using the training set.
* Predict the labels of the test set. What is the accuracy on the test set?

In [77]:
# put your code here
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
import numpy as np

labels = data.iloc[:, -1].astype(str).str.strip()
y_logit = np.where(labels == "Mine", 1, 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_logit, test_size=0.15, random_state=0
)

X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

logit_model = sm.Logit(y_train, X_train_const)
result = logit_model.fit()

y_prob = result.predict(X_test_const)
y_pred = np.where(y_prob >= 0.5, 1, 0)

test_accuracy = np.mean(y_pred == y_test)
print("Logistic Regression Test Accuracy:", test_accuracy)

Optimization terminated successfully.
         Current function value: 0.383414
         Iterations 7
Logistic Regression Test Accuracy: 0.75


&#9989; **Do This**: In the cell below, figure our how the accuracy compares between the training set and test set for this logistic regression model? Discuss your observations with your group.

In [82]:
# Put your code here
train_prob = result.predict(X_train_const)
train_pred = np.where(train_prob >= 0.5, 1, 0)
train_accuracy = np.mean(train_pred == y_train)

print("Logistic Regression Training Accuracy:", train_accuracy)
print("Logistic Regression Test Accuracy:", test_accuracy)

Logistic Regression Training Accuracy: 0.8068181818181818
Logistic Regression Test Accuracy: 0.75


----
&#9989; **Do This**: Now, Train your custom percentron class with the training set. 
* Make sure your labels are correctly defined for the perceptron classifier.
* Use the features in the test set in your perceptron prediction function to predict the labels. 
* What is the accuracy of your perceptron classifier on the test set? How does it compare to the logistic regression model?

In [84]:
y_perc = np.where(labels == "Mine", 1, -1)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y_perc, test_size=0.15, random_state=0
)

perc_model = MyPerceptron(learning_rate=0.01, n_iters=1000)

perc_model.fit(X_train_p, y_train_p)

perc_pred = perc_model.predict(X_test_p)

perc_accuracy = np.mean(perc_pred == y_test_p)

print("Custom Perceptron Test Accuracy:", perc_accuracy)

Custom Perceptron Test Accuracy: 0.71875


---
&#9989; **Do This**: Provide a short discussion of the comparison between the results from the different classifiers.


The logistic regression model and perceptron performed similarly because both are linear classifiers. Logistic regression is often more stable because it uses probabilities and minimizes a continuous loss function. The perceptron updates weights only when errors occur, which can make it more sensitive to parameter choices like learning rate and iterations. Logistic regression usually generalizes slightly better on unseen data

-----
### Congratulations, we're done!

Now, you just need to submit this assignment by uploading it to the course <a href="https://d2l.msu.edu/">Desire2Learn</a> web page for today's submission folder (Don't forget to add your names in the first cell).


&#169; Copyright 2026, The Department of Computational Mathematics, Science and Engineering; Michigan State University